In [ ]:
"""
Module Python équivalent à library.erl
Traite les configurations, les ensembles, et les opérations de cliques
"""
import math
from typing import List, Tuple, Set, Dict, Any, Union
import itertools
import struct

# Import pour la compatibilité avec l'original
try:
    from codinglist import coding_list
except ImportError:
    # Fallback si le module n'existe pas
    def coding_list(n: int) -> List[int]:
        return list(range(n))

def index(element: Any, lst: List[Any]) -> int:
    """
    Retourne l'index d'un élément dans une liste
    Équivalent à index/2 en Erlang
    """
    try:
        return lst.index(element)
    except ValueError:
        return -1

def index2(element: Any, lst: List[Any], n: int = 0) -> int:
    """
    Version récursive de index
    Équivalent à index2/3 en Erlang
    """
    if not lst:
        return -1
    if lst[0] == element:
        return n
    return index2(element, lst[1:], n + 1)

def puiss(n: int) -> int:
    """
    Calcule 2^n (puissance de 2)
    Équivalent à puiss/1 en Erlang
    """
    if n == 0:
        return 1
    return 2 * puiss(n - 1)

def intofindex(i: int) -> int:
    """
    Retourne 2^i
    Équivalent à intofindex/1 en Erlang
    """
    return puiss(i)

def symboliclique(S: Set[int], Refvec: List[int]) -> int:
    """
    Convertit un ensemble en représentation symbolique
    Équivalent à symboliclique/2 en Erlang
    """
    L = list(S)
    total = 0
    for element in L:
        idx = index(element, Refvec)
        if idx >= 0:
            total += intofindex(idx)
    return total

def select(zip_list: List[Tuple[int, int]]) -> List[int]:
    """
    Sélectionne les éléments où le premier élément du tuple est 1
    Équivalent à select/1 en Erlang
    """
    result = []
    for bit, element in zip_list:
        if bit == 1:
            result.append(element)
    return result

def ensclique(C: int, Refvec: List[int]) -> Set[int]:
    """
    Convertit un entier en ensemble d'éléments
    Équivalent à ensclique/2 en Erlang
    """
    size = len(Refvec)
    # Convertir l'entier en vecteur binaire
    V = inttovect(C, size)
    V_reversed = list(reversed(V))  # Equivalent à lists:reverse
    
    # Zipper avec Refvec
    Z = list(zip(V_reversed, Refvec))
    
    # Sélectionner les éléments
    selected = select(Z)
    
    return set(selected)

def vecttoint(V: List[int]) -> int:
    """
    Convertit un vecteur binaire en entier
    Équivalent à vecttoint/1 en Erlang
    """
    return vecttoint2(V, 0)

def vecttoint2(V: List[int], N: int) -> int:
    """
    Helper récursif pour vecttoint
    Équivalent à vecttoint2/2 en Erlang
    """
    if N == 0:
        V1 = list(reversed(V))
    else:
        V1 = V
    
    if not V1:
        return 0
    
    if V1[0] == 1:
        return puiss(N) + vecttoint2(V1[1:], N + 1)
    else:  # V1[0] == 0
        return vecttoint2(V1[1:], N + 1)

def inttovect(N: int, Size: int) -> List[int]:
    """
    Convertit un entier en vecteur binaire de taille donnée
    Équivalent à inttovect/2 en Erlang
    """
    # Utiliser le format binaire
    binary_str = format(N, f'0{Size}b')
    return [int(bit) for bit in binary_str]

def countSetBits(N: int) -> int:
    """
    Compte le nombre de bits à 1 dans un entier
    Équivalent à countSetBits/1 en Erlang
    """
    # Méthode 1: Avec format binaire
    binary_str = bin(N)[2:]  # Retirer le '0b' du début
    return sum(int(bit) for bit in binary_str)
    
    # Méthode alternative plus rapide:
    # return bin(N).count('1')

def symbolictreated(T: int, Refvec: List[int]) -> int:
    """
    Alias pour symboliclique (pour la compatibilité)
    Équivalent à symbolictreated/2 en Erlang
    """
    return symboliclique(ensclique(T, Refvec), Refvec)

def enstreated(Ts: int, Refvec: List[int]) -> Set[int]:
    """
    Alias pour ensclique (pour la compatibilité)
    Équivalent à enstreated/2 en Erlang
    """
    return ensclique(Ts, Refvec)

def makeConfiguration(Ts: int, C: int, Refvec: List[int]) -> bytes:
    """
    Crée une configuration à partir de deux entiers
    Équivalent à makeConfiguration/3 en Erlang
    """
    Size = len(Refvec)
    
    # Vérifier que les valeurs tiennent dans la taille
    max_val = 2**Size - 1
    if Ts > max_val or C > max_val:
        raise ValueError(f"Values exceed maximum for size {Size}")
    
    # Packer les deux entiers en bytes
    # Utiliser '>I' pour big-endian (comme Erlang)
    try:
        # Essayer avec struct
        result = struct.pack(f'>II', Ts, C)
    except:
        # Fallback: méthode manuelle
        ts_bytes = Ts.to_bytes(Size // 8, byteorder='big')
        c_bytes = C.to_bytes(Size // 8, byteorder='big')
        result = ts_bytes + c_bytes
    
    return result

def split_Conf(Conf: bytes, Refvec: List[int]) -> Tuple[int, int]:
    """
    Sépare une configuration en deux entiers
    Équivalent à split_Conf/2 en Erlang
    """
    Size = len(Refvec)
    bytes_per_int = max(1, (Size + 7) // 8)  # Arrondir au nombre d'octets nécessaire
    
    try:
        # Essayer avec struct
        format_str = f'>II'
        expected_len = 8  # 2 * 4 bytes pour 2 entiers 32-bit
        if len(Conf) < expected_len:
            # Padding si nécessaire
            Conf = Conf.ljust(expected_len, b'\x00')
        Fst, Snd = struct.unpack(format_str, Conf[:expected_len])
    except:
        # Fallback: méthode manuelle
        if len(Conf) < 2 * bytes_per_int:
            # Padding
            Conf = Conf.ljust(2 * bytes_per_int, b'\x00')
        
        Fst_bytes = Conf[:bytes_per_int]
        Snd_bytes = Conf[bytes_per_int:2 * bytes_per_int]
        
        Fst = int.from_bytes(Fst_bytes, byteorder='big')
        Snd = int.from_bytes(Snd_bytes, byteorder='big')
    
    return Fst, Snd

def first(Conf: bytes, Refvec: List[int]) -> int:
    """
    Retourne la première partie d'une configuration
    Équivalent à first/2 en Erlang
    """
    Fst, _ = split_Conf(Conf, Refvec)
    return Fst

def second(Conf: bytes, Refvec: List[int]) -> int:
    """
    Retourne la seconde partie d'une configuration
    Équivalent à second/2 en Erlang
    """
    _, Snd = split_Conf(Conf, Refvec)
    return Snd

def is_Terminal(Conf: bytes, Refvec: List[int]) -> bool:
    """
    Vérifie si une configuration est terminale
    Équivalent à is_Terminal/2 en Erlang
    """
    Fst = first(Conf, Refvec)
    Snd = second(Conf, Refvec)
    
    # Condition: (Snd & Fst) == Snd
    # Si tous les bits de Snd sont dans Fst, alors c'est terminal
    return (Snd & Fst) == Snd

def choose(Fst: int, Snd: int, Refvec: List[int]) -> int:
    """
    Choisit un élément selon certaines règles
    Équivalent à choose/3 en Erlang
    """
    Size = len(Refvec)
    
    # Calculer V = ~(Fst & Snd)
    intersection = Fst & Snd
    # Inverser les bits pour la taille donnée
    mask = (1 << Size) - 1
    V = (~intersection) & mask
    
    # Convertir en vecteur
    V_bits = inttovect(V, Size)
    
    # Calculer N1
    N1 = vecttoint(V_bits)
    
    # Calculer EL = N1 & Snd
    EL = N1 & Snd
    
    # Obtenir l'ensemble
    Ens = ensclique(EL, Refvec)
    
    if not Ens:
        raise ValueError("Empty set returned from ensclique")
    
    # Retourner le premier élément
    return next(iter(Ens))

def split(Conf: bytes, Refvec: List[int]) -> Tuple[bytes, bytes]:
    """
    Divise une configuration en deux sous-configurations
    Équivalent à split/2 en Erlang
    """
    Treated = first(Conf, Refvec)
    Clique = second(Conf, Refvec)
    
    # Choisir un élément E
    E = choose(Treated, Clique, Refvec)
    print(f"e={E}")  # Equivalent à io:format
    
    # Obtenir l'ensemble R
    R_set = ensclique(Clique, Refvec)
    R_list = list(R_set)
    
    # Filtrer les éléments où X & E == 0
    Z = [X for X in R_list if (X & E) == 0]
    
    # Créer les ensembles Zero et One
    Zero_set = set([E] + Z)
    One_set = R_set - set([E])
    
    # Calculer les valeurs symboliques
    Mate = symboliclique(Zero_set, Refvec)
    Rival = symboliclique(One_set, Refvec)
    
    # Créer le nouvel ensemble traité
    Treated_set = enstreated(Treated, Refvec)
    Treated_set.add(E)
    Ts = symboliclique(Treated_set, Refvec)
    
    # Créer les nouvelles configurations
    Rs1 = makeConfiguration(Ts, Mate, Refvec)
    Rs2 = makeConfiguration(Ts, Rival, Refvec)
    
    return Rs1, Rs2

def displayOfConf(Conf: bytes, Refvec: List[int]) -> Tuple[List[int], List[int]]:
    """
    Affiche une configuration de manière lisible
    Équivalent à displayOfConf/2 en Erlang
    """
    Fst = first(Conf, Refvec)
    Snd = second(Conf, Refvec)
    
    Treated_set = ensclique(Fst, Refvec)
    Clique_set = ensclique(Snd, Refvec)
    
    Treated = sorted(list(Treated_set))
    Clique = sorted(list(Clique_set))
    
    return (Treated, Clique)

def getInitialConf(Refvec: List[int]) -> bytes:
    """
    Retourne la configuration initiale
    Équivalent à getInitialConf/1 en Erlang
    """
    empty_set = set()
    all_set = set(Refvec)
    
    Ts = symboliclique(empty_set, Refvec)
    C = symboliclique(all_set, Refvec)
    
    return makeConfiguration(Ts, C, Refvec)

def displayOnChess(Conf: bytes, Refvec: List[int]) -> List[int]:
    """
    Affiche la configuration sous forme d'échiquier
    Équivalent à displayOnChess/2 en Erlang
    """
    _, Clique = displayOfConf(Conf, Refvec)
    Chess = [index(E, Refvec) for E in Clique]
    return Chess

# Fonctions utilitaires supplémentaires
def from_list(lst: List[Any]) -> Set[Any]:
    """
    Simule sets:from_list/1 d'Erlang
    """
    return set(lst)

def to_list(s: Set[Any]) -> List[Any]:
    """
    Simule sets:to_list/1 d'Erlang
    """
    return list(s)

def subtract(set1: Set[Any], set2: Set[Any]) -> Set[Any]:
    """
    Simule sets:subtract/2 d'Erlang
    """
    return set1 - set2

def add_element(element: Any, s: Set[Any]) -> Set[Any]:
    """
    Simule sets:add_element/2 d'Erlang
    """
    return s.union({element})

def filter(func, lst: List[Any]) -> List[Any]:
    """
    Simule lists:filter/2 d'Erlang
    """
    return [x for x in lst if func(x)]

def append(lst1: List[Any], lst2: List[Any]) -> List[Any]:
    """
    Simule lists:append/2 d'Erlang
    """
    return lst1 + lst2

def reverse(lst: List[Any]) -> List[Any]:
    """
    Simule lists:reverse/1 d'Erlang
    """
    return list(reversed(lst))

def map(func, lst: List[Any]) -> List[Any]:
    """
    Simule lists:map/2 d'Erlang
    """
    return [func(x) for x in lst]

def sum(lst: List[int]) -> int:
    """
    Simule lists:sum/1 d'Erlang
    """
    return sum(lst)

# Tests unitaires
def test_library():
    """Fonction de test pour vérifier les fonctions"""
    
    print("Testing library module...")
    
    # Créer un Refvec
    Refvec = coding_list(4)
    print(f"Refvec: {Refvec}")
    
    # Test puiss
    assert puiss(0) == 1
    assert puiss(3) == 8
    
    # Test index
    assert index(2, Refvec) == 2
    assert index(10, Refvec) == -1
    
    # Test symboliclique et ensclique
    test_set = {1, 3}
    sym = symboliclique(test_set, Refvec)
    print(f"Set {test_set} -> symbolic: {sym}")
    
    restored = ensclique(sym, Refvec)
    print(f"Symbolic {sym} -> set: {restored}")
    assert test_set == restored
    
    # Test inttovect et vecttoint
    test_int = 5  # 0101 en binaire pour size=4
    vect = inttovect(test_int, 4)
    print(f"Int {test_int} -> vect: {vect}")
    
    back_int = vecttoint(vect)
    print(f"Vect {vect} -> int: {back_int}")
    assert test_int == back_int
    
    # Test countSetBits
    assert countSetBits(5) == 2  # 0101 a 2 bits à 1
    assert countSetBits(7) == 3  # 0111 a 3 bits à 1
    
    # Test makeConfiguration et split_Conf
    Ts = 3  # 0011
    C = 5   # 0101
    conf = makeConfiguration(Ts, C, Refvec)
    print(f"Configuration Ts={Ts}, C={C} -> bytes: {conf.hex()}")
    
    Ts_back, C_back = split_Conf(conf, Refvec)
    print(f"Bytes -> Ts={Ts_back}, C={C_back}")
    assert Ts == Ts_back and C == C_back
    
    # Test is_Terminal
    terminal_conf = makeConfiguration(7, 3, Refvec)  # 7=0111, 3=0011 (3⊂7)
    non_terminal_conf = makeConfiguration(3, 7, Refvec)  # 3=0011, 7=0111 (7⊄3)
    
    assert is_Terminal(terminal_conf, Refvec) == True
    assert is_Terminal(non_terminal_conf, Refvec) == False
    
    print("All tests passed!")

def demo():
    """Démonstration des fonctionnalités"""
    
    print("\n=== DEMONSTRATION ===\n")
    
    # Initialiser
    Refvec = coding_list(6)
    print(f"Reference vector: {Refvec}")
    
    # Créer configuration initiale
    initial_conf = getInitialConf(Refvec)
    print(f"Initial configuration created")
    
    # Afficher la configuration
    treated, clique = displayOfConf(initial_conf, Refvec)
    print(f"Treated: {treated}")
    print(f"Clique: {clique}")
    
    # Vérifier si terminal
    is_term = is_Terminal(initial_conf, Refvec)
    print(f"Is terminal? {is_term}")
    
    if not is_term:
        # Diviser la configuration
        print("\nSplitting configuration...")
        conf1, conf2 = split(initial_conf, Refvec)
        
        print("Split successful!")
        print(f"Configuration 1: {displayOfConf(conf1, Refvec)}")
        print(f"Configuration 2: {displayOfConf(conf2, Refvec)}")
    
    # Afficher sur échiquier
    chess_board = displayOnChess(initial_conf, Refvec)
    print(f"\nChess board representation: {chess_board}")

if __name__ == "__main__":
    # Exécuter les tests
    test_library()
    
    # Exécuter la démo
    demo()